In [1]:
import sys; sys.path.append('..')
import joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt, shap
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/wallets/transaction_dataset.csv")
df.columns = df.columns.str.strip()
df = df.drop(columns=["Unnamed: 0", "Index"], errors="ignore").drop_duplicates(subset="Address")
y = df["FLAG"]
X = df.drop(columns=["FLAG", "Address"]).select_dtypes("number")
X = X.loc[:, X.nunique(dropna=False) > 1]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

bundle = joblib.load("../models/wallet-xgb-v1.joblib")
final, top20 = bundle["pipeline"], bundle["features"]
clf, imp = final.named_steps["clf"], final.named_steps["impute"]
X_te_i = pd.DataFrame(imp.transform(X_te[top20]), columns=top20, index=X_te.index)
X_tr_i = pd.DataFrame(imp.transform(X_tr[top20]), columns=top20, index=X_tr.index)
print(X_te_i.shape)

(1964, 20)


In [2]:
explainer = shap.TreeExplainer(clf)
sv = explainer(X_te_i)

shap.plots.beeswarm(sv, max_display=15, show=False)
plt.savefig("../results/m3_shap_beeswarm.png", dpi=200, bbox_inches="tight"); plt.close()

i = int(np.argmax(clf.predict_proba(X_te_i)[:, 1]))   # l'adresse la plus suspecte
shap.plots.waterfall(sv[i], max_display=10, show=False)
plt.savefig("../results/m3_shap_waterfall.png", dpi=200, bbox_inches="tight"); plt.close()

In [3]:
from lime.lime_tabular import LimeTabularExplainer

lime_exp = LimeTabularExplainer(X_tr_i.values, feature_names=top20,
                                class_names=["normal", "fraude"], mode="classification",
                                discretize_continuous=True, random_state=42)
predict_fn = lambda a: clf.predict_proba(pd.DataFrame(a, columns=top20))

def lime_vector(x_row, num_features=len(top20)):
    e = lime_exp.explain_instance(x_row.values, predict_fn,
                                  num_features=num_features, num_samples=1000)
    v = np.zeros(len(top20))
    for j, w in e.as_map()[1]:
        v[j] = w
    return v

def shap_vector(x_row):
    return explainer(x_row.to_frame().T).values[0]

In [4]:
rng = np.random.default_rng(42)
baseline = X_tr_i.median()
pred = clf.predict_proba(X_te_i)[:, 1]
detected = X_te_i[(pred >= 0.5) & (y_te.values == 1)].head(100)   # vraies fraudes détectées

def deletion_drop(x_row, ranking, k):
    x_mod = x_row.copy()
    cols = [top20[j] for j in ranking[:k]]
    x_mod[cols] = baseline[cols]
    p0 = clf.predict_proba(x_row.to_frame().T)[0, 1]
    p1 = clf.predict_proba(x_mod.to_frame().T)[0, 1]
    return p0 - p1

results = {"shap": [], "lime": [], "random": []}
for _, x in detected.iterrows():
    rank = {"shap": np.argsort(-np.abs(shap_vector(x))),
            "lime": np.argsort(-np.abs(lime_vector(x))),
            "random": rng.permutation(len(top20))}
    for m in results:
        results[m].append([deletion_drop(x, rank[m], k) for k in range(1, 11)])

for m, v in results.items():
    plt.plot(range(1, 11), np.mean(v, axis=0), marker="o", label=m.upper())
plt.xlabel("k features retirées"); plt.ylabel("Baisse moyenne de probabilité")
plt.legend(); plt.grid(alpha=.3)
plt.savefig("../results/m3_fidelity.png", dpi=200, bbox_inches="tight")

In [5]:
from scipy.stats import spearmanr

def stability(explain_fn, x_row, noise=0.02, n=10, k=5):
    base = np.abs(explain_fn(x_row))
    top_base = set(np.argsort(-base)[:k])
    jac, rho = [], []
    for _ in range(n):
        x_noisy = x_row * (1 + rng.normal(0, noise, size=len(x_row)))
        e = np.abs(explain_fn(x_noisy))
        top = set(np.argsort(-e)[:k])
        jac.append(len(top & top_base) / len(top | top_base))
        rho.append(spearmanr(base, e).statistic)
    return np.mean(jac), np.mean(rho)

rows = []
for noise in [0.01, 0.02, 0.05]:
    for m, fn in [("SHAP", shap_vector), ("LIME", lime_vector)]:
        s = [stability(fn, x, noise=noise) for _, x in detected.head(30).iterrows()]
        rows.append({"bruit": noise, "méthode": m,
                     "jaccard_top5": np.mean([a for a, _ in s]),
                     "spearman": np.mean([b for _, b in s])})
stab = pd.DataFrame(rows)
stab.to_csv("../results/m3_stability.csv", index=False)
stab

,bruit,méthode,jaccard_top5,spearman
0,0.01,SHAP,0.958889,0.952175
1,0.01,LIME,0.596032,0.497173
2,0.02,SHAP,0.952222,0.949714
3,0.02,LIME,0.568056,0.475083
4,0.05,SHAP,0.934762,0.938857
5,0.05,LIME,0.606005,0.466075
